In [ ]:
#Pick cue trials from target
import os
import json
import mne

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

EPOCHS_DIR = r"F:\epochs_ITC"
EPOCHS_FNAME_PATTERN = "{sub}-raw-ica-reject-ERP-epo.fif"

OUT_CUE_DIR = r"F:\cue_epoch"
OUT_TARG_DIR = r"F:\target_epoch"

os.makedirs(OUT_CUE_DIR, exist_ok=True)
os.makedirs(OUT_TARG_DIR, exist_ok=True)

#select
def pick_keys(event_id: dict, prefix: str, exclude=("remove",)):
    keys = []
    for k in event_id.keys():
        if any(k == ex for ex in exclude):
            continue
        if k.startswith(prefix):
            keys.append(k)
    return keys


# 3) MAIN
log = {}
errors = {}

print("=== Split mixed epochs into cue-epochs and targ-epochs (per subject) ===")
print("Input dir :", EPOCHS_DIR)
print("-----------------------------------------------------------------------")

for sub in subjects:
    try:
        in_path = os.path.join(EPOCHS_DIR, EPOCHS_FNAME_PATTERN.format(sub=sub))
        if not os.path.exists(in_path):
            raise FileNotFoundError(f"Missing epochs file: {in_path}")

        print(f"\n[{sub}] Loading: {in_path}")
        epochs_all = mne.read_epochs(in_path, preload=True, verbose=False)

        # Determine keys
        cue_keys = pick_keys(epochs_all.event_id, prefix="cue/")
        targ_keys = pick_keys(epochs_all.event_id, prefix="targ/")

        if len(cue_keys) == 0:
            raise RuntimeError(f"No cue keys found in event_id. Keys sample: {list(epochs_all.event_id)[:20]}")
        if len(targ_keys) == 0:
            raise RuntimeError(f"No targ keys found in event_id. Keys sample: {list(epochs_all.event_id)[:20]}")

        # Subset epochs
        epochs_cue = epochs_all[cue_keys]
        epochs_targ = epochs_all[targ_keys]

        if len(epochs_cue) == 0:
            raise RuntimeError(f"cue_keys exist but selected 0 trials. cue_keys={cue_keys}")
        if len(epochs_targ) == 0:
            raise RuntimeError(f"targ_keys exist but selected 0 trials. targ_keys={targ_keys}")

        # Save
        out_cue = os.path.join(OUT_CUE_DIR, f"{sub}_cue-epo.fif")
        out_targ = os.path.join(OUT_TARG_DIR, f"{sub}_targ-epo.fif")

        epochs_cue.save(out_cue, overwrite=True)
        epochs_targ.save(out_targ, overwrite=True)

        print(f"[{sub}] cue  trials: {len(epochs_cue)}  -> {out_cue}")
        print(f"[{sub}] targ trials: {len(epochs_targ)}  -> {out_targ}")

        log[sub] = {
            "in_path": in_path,
            "cue_keys": cue_keys,
            "targ_keys": targ_keys,
            "n_cue_trials": int(len(epochs_cue)),
            "n_targ_trials": int(len(epochs_targ)),
            "out_cue": out_cue,
            "out_targ": out_targ,
            "tmin": float(epochs_all.tmin),
            "tmax": float(epochs_all.tmax),
        }

    except Exception as e:
        errors[sub] = str(e)
        print(f"[{sub}] ERROR: {e}")


In [ ]:
#Sanity check
import os
import mne

CUE_DIR = r"F:\cue_epoch"
files = sorted([f for f in os.listdir(CUE_DIR) if f.endswith("-epo.fif")])[:5]

print("Checking cue epochs files:\n")

for fname in files:
    fpath = os.path.join(CUE_DIR, fname)
    epochs = mne.read_epochs(fpath, preload=False, verbose=False)

    keys = list(epochs.event_id.keys())

    print(f"--- {fname} ---")
    print(f"n_trials : {len(epochs)}")
    print("event_id keys:")
    for k in keys:
        print(" ", k)

    # 简单逻辑检查
    has_targ = any(k.startswith("targ/") for k in keys)
    has_cue  = any(k.startswith("cue/") for k in keys)
    has_other = any(
        (not k.startswith("cue/")) for k in keys
    )

    if has_targ:
        print("ERROR！！！！！！！！！！！: targ event found!")
    elif not has_cue:
        print("ERROR！！！！！！！！！！！: no cue event found!")
    elif has_other:
        print("WARNING: non-cue event found!")
    else:
        print("Yoo~Hoo~! OK: cue-only epochs")

    print()

-Compute Single Trial TFR for cue-only (without power baseline correction

In [ ]:
subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

CUE_EPOCH_DIR = r"F:\cue_epoch"
CUE_FNAME_PATTERN = "{sub}_cue-epo.fif"

OUT_DIR = r"F:\cue_single_trial_TFR_noBaeslineCorrection"
os.makedirs(OUT_DIR, exist_ok=True)

freqs = np.arange(8, 30, 1)  
n_cycles = 6                
decim = 1
tmax_compute = 1.0          
use_fft = True                       
verbose_mne = False

def save_tfr(sub: str, tfr: mne.time_frequency.EpochsTFR):
    power_path = os.path.join(OUT_DIR, f"{sub}_cue_power_noBC.npy")
    meta_path = os.path.join(OUT_DIR, f"{sub}_cue_meta_noBC.npz")

    np.save(power_path, tfr.data.astype(np.float32))

    meta = dict(
        subject=sub,
        sfreq=float(tfr.info["sfreq"]),
        ch_names=np.array(tfr.ch_names, dtype=object),
        freqs=tfr.freqs.astype(np.float64),
        times=tfr.times.astype(np.float64),
        method="morlet",
        n_cycles=n_cycles if np.isscalar(n_cycles) else np.array(n_cycles, dtype=np.float64),
        decim=int(decim),
        average=False,
        return_itc=False,
        use_fft=bool(use_fft),
        tmin=float(tfr.times[0]),
        tmax=float(tfr.times[-1]),
        data_shape=np.array(tfr.data.shape, dtype=np.int64),
        note="NO baseline correction applied here",
    )
    np.savez(meta_path, **meta)

    return power_path, meta_path


#MAIN LOOP
errors = {}
saved = []

print("=== Cue single-trial TFR (NO baseline correction) ===")
print("Input cue epochs dir :", CUE_EPOCH_DIR)
print("Output dir          :", OUT_DIR)
print(f"freqs: {freqs[0]}..{freqs[-1]} Hz (n={len(freqs)}), n_cycles={n_cycles}, decim={decim}, tmax={tmax_compute}, use_fft={use_fft}")
print("-----------------------------------------------------")

for sub in subjects:
    try:
        fpath = os.path.join(CUE_EPOCH_DIR, CUE_FNAME_PATTERN.format(sub=sub))
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing cue epochs file: {fpath}")

        print(f"\n[{sub}] Loading cue epochs ...")
        epochs_cue = mne.read_epochs(fpath, preload=True, verbose=verbose_mne)
        epochs_cue = epochs_cue.copy().pick_types(eeg=True, eog=False, ecg=False, stim=False, misc=False)

        # sanity: should only contain cue keys
        keys = list(epochs_cue.event_id.keys())
        if not all(k.startswith("cue/") for k in keys):
            bad = [k for k in keys if not k.startswith("cue/")]
            print(f"[{sub}] WARNING: non-cue event_id keys found: {bad}")

        if len(epochs_cue) == 0:
            raise RuntimeError("cue epochs has 0 trials.")

        print(f"[{sub}] epochs: n_trials={len(epochs_cue)}, tmin={epochs_cue.tmin:.3f}, tmax={epochs_cue.tmax:.3f}, n_ch={len(epochs_cue.ch_names)}")

        print(f"[{sub}] Computing TFR via epochs.compute_tfr(...) ...")
        power_cue = epochs_cue.compute_tfr(
            method="morlet",
            freqs=freqs,
            n_cycles=n_cycles,
            tmin=epochs_cue.tmin,
            tmax=tmax_compute,
            average=False,
            return_itc=False,
            decim=decim,
            use_fft=use_fft,
            verbose=verbose_mne,
        )

        # DO NOT apply baseline here.

        power_path, meta_path = save_tfr(sub, power_cue)
        saved.append(sub)

        print(f"[{sub}] Saved:\n  {power_path}\n  {meta_path}")
        print(f"[{sub}] TFR shape: {power_cue.data.shape} (n_trials, n_ch, n_freq, n_times)")
        print(f"[{sub}] times range: {float(power_cue.times[0])} .. {float(power_cue.times[-1])}")

    except Exception as e:
        errors[sub] = str(e)
        print(f"[{sub}] ERROR: {e}")

print("\n================ SUMMARY ================")
print(f"Saved: {len(saved)} / {len(subjects)}")

if errors:
    err_path = os.path.join(OUT_DIR, "cue_tfr_noBC_errors.json")
    with open(err_path, "w", encoding="utf-8") as f:
        json.dump(errors, f, indent=2, ensure_ascii=False)
    print(f"Errors: {len(errors)} -> {err_path}")
else:
    print("No errors.")

In [ ]:
#Check TFR contain the right time window
import os
import numpy as np

TFR_DIR = r"F:\cue_single_trial_TFR_noBaeslineCorrection"
baseline = (-0.2, 0.0)

meta_files = sorted([
    f for f in os.listdir(TFR_DIR)
    if f.endswith("_cue_meta_noBC.npz")
])[:5]

print("Checking cue TFR baseline window coverage (-0.2, 0):\n")

for mf in meta_files:
    meta_path = os.path.join(TFR_DIR, mf)
    meta = np.load(meta_path, allow_pickle=True)

    times = meta["times"]  # 1D array
    tmin, tmax = float(times[0]), float(times[-1])

    has_baseline = (tmin <= baseline[0]) and (tmax >= baseline[1])
    n_baseline_pts = ((times >= baseline[0]) & (times <= baseline[1])).sum()

    print(f"--- {mf} ---")
    print(f"time range : {tmin:.3f} .. {tmax:.3f}")
    print(f"baseline pts in [-0.2,0]: {n_baseline_pts}")

    if has_baseline and n_baseline_pts > 0:
        print("OK: baseline window present\n")
    else:
        print("ERROR: baseline window NOT present\n")


In [ ]:
#Compute baseline
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

TFR_DIR = r"F:\cue_single_trial_TFR_noBaeslineCorrection"
OUT_DIR = r"F:\cue_global_baseline_Bglobal"
os.makedirs(OUT_DIR, exist_ok=True)

POWER_FNAME = "{sub}_cue_power_noBC.npy"
META_FNAME = "{sub}_cue_meta_noBC.npz"

#BASELINE SETTINGS
baseline = (-0.2, 0.0)
tol = 0.002  # seconds; ~2 ms tolerance

#CORE: compute B_global(ch,f)
def compute_B_global_from_saved_tfr(power_path: str, meta_path: str, baseline=(-0.2, 0.0), tol=0.002):
    meta = np.load(meta_path, allow_pickle=True)
    times = meta["times"].astype(float)

    power = np.load(power_path)  # float32 ok
    if power.ndim != 4:
        raise RuntimeError(f"Unexpected power ndim={power.ndim}, expected 4. power_path={power_path}")

    tmin, tmax = float(times[0]), float(times[-1])

    # Coverage check with tolerance
    if tmin > baseline[0] + tol:
        raise ValueError(f"Times start at {tmin:.6f}, does NOT cover baseline start {baseline[0]:.3f} (tol={tol}).")
    if tmax < baseline[1] - tol:
        raise ValueError(f"Times end at {tmax:.6f}, does NOT cover baseline end {baseline[1]:.3f} (tol={tol}).")

    # Baseline mask (inclusive)
    tmask = (times >= baseline[0] - tol) & (times <= baseline[1] + tol)
    n_base_pts = int(tmask.sum())
    if n_base_pts <= 0:
        raise RuntimeError(f"No baseline points found with mask. times range={tmin}..{tmax}, baseline={baseline}, tol={tol}")

    # Mean over trials and time within baseline -> (n_ch, n_freq)
    B_global = power[..., tmask].mean(axis=(0, 3))

    # Numerical safety
    eps = np.finfo(float).eps
    B_global = np.maximum(B_global, eps)

    info = {
        "tmin": tmin,
        "tmax": tmax,
        "baseline": baseline,
        "tol": tol,
        "n_baseline_pts": n_base_pts,
        "times_at_baseline_start": float(times[np.where(tmask)[0][0]]),
        "times_at_baseline_end": float(times[np.where(tmask)[0][-1]]),
        "data_shape": tuple(power.shape),
    }
    return B_global, meta, info


def save_B_global(sub: str, B_global: np.ndarray, meta_in, info: dict, out_dir: str):
    out_npy = os.path.join(out_dir, f"{sub}_Bglobal_chxf.npy")
    out_meta = os.path.join(out_dir, f"{sub}_Bglobal_meta.npz")

    np.save(out_npy, B_global.astype(np.float32))

    meta_out = dict(
        subject=sub,
        ch_names=meta_in["ch_names"],
        freqs=meta_in["freqs"].astype(np.float64),
        baseline=np.array(info["baseline"], dtype=np.float64),
        tol=np.array(info["tol"], dtype=np.float64),
        n_baseline_pts=np.array(info["n_baseline_pts"], dtype=np.int64),
        baseline_start_time=np.array(info["times_at_baseline_start"], dtype=np.float64),
        baseline_end_time=np.array(info["times_at_baseline_end"], dtype=np.float64),
        tfr_times_start=np.array(info["tmin"], dtype=np.float64),
        tfr_times_end=np.array(info["tmax"], dtype=np.float64),
        bglobal_shape=np.array(B_global.shape, dtype=np.int64),
        source_power_shape=np.array(info["data_shape"], dtype=np.int64),
        note="B_global(ch,f) computed from cue single-trial TFR baseline window; mean over trials and baseline-time",
    )
    np.savez(out_meta, **meta_out)
    return out_npy, out_meta


# 4) QUICK-PLOT (random subject)
def quick_plot_baseline_distribution(sub: str, power_path: str, meta_path: str, baseline=(-0.2, 0.0), tol=0.002, out_dir: str = None):
    meta = np.load(meta_path, allow_pickle=True)
    times = meta["times"].astype(float)
    freqs = meta["freqs"].astype(float)

    power = np.load(power_path).astype(np.float64)
    tmask = (times >= baseline[0] - tol) & (times <= baseline[1] + tol)
    base_block = power[..., tmask]  # (trial, ch, f, tbase)

    # Flatten for histogram
    flat = base_block.ravel()
    flat = flat[np.isfinite(flat)]

    # Compute B_global too
    B_global = base_block.mean(axis=(0, 3))  # (ch,f)
    bflat = B_global.ravel()
    bflat = bflat[np.isfinite(bflat)]

    # Robust quantiles for outlier screen
    q = np.quantile(flat, [0.0, 0.001, 0.01, 0.5, 0.99, 0.999, 1.0])

    plt.figure()
    plt.hist(flat, bins=200)  # default color ok
    plt.title(f"{sub} cue baseline power distribution\nbaseline={baseline}, pts={tmask.sum()}, freqs={freqs[0]}-{freqs[-1]}Hz")
    plt.xlabel("Power (linear scale)")
    plt.ylabel("Count")

    # Annotate quantiles
    txt = (
        "quantiles of baseline power:\n"
        f"min={q[0]:.3e}\n"
        f"0.1%={q[1]:.3e}\n"
        f"1%={q[2]:.3e}\n"
        f"median={q[3]:.3e}\n"
        f"99%={q[4]:.3e}\n"
        f"99.9%={q[5]:.3e}\n"
        f"max={q[6]:.3e}\n"
        f"n={flat.size}"
    )
    plt.gca().text(0.98, 0.98, txt, transform=plt.gca().transAxes,
                   ha="right", va="top")

    if out_dir is None:
        out_dir = os.path.dirname(power_path)

    out_png = os.path.join(out_dir, f"{sub}_baseline_qc.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

    # Second plot: B_global distribution (ch x f)
    plt.figure()
    plt.hist(bflat, bins=200)
    plt.title(f"{sub} B_global(ch,f) distribution (mean over trials & baseline time)")
    plt.xlabel("B_global (linear power)")
    plt.ylabel("Count")
    plt.tight_layout()
    out_png2 = os.path.join(out_dir, f"{sub}_Bglobal_qc.png")
    plt.savefig(out_png2, dpi=150)
    plt.close()

    return out_png, out_png2, {
        "baseline_quantiles": q.tolist(),
        "n_baseline_values": int(flat.size),
        "n_bglobal_values": int(bflat.size),
    }


#MAIN LOOP
errors = {}
summary = {}

print("=== Step: Compute & Save B_global(ch,f) per subject ===")
print("TFR_DIR:", TFR_DIR)
print("OUT_DIR:", OUT_DIR)
print(f"baseline={baseline}, tol={tol}")
print("------------------------------------------------------")

for sub in subjects:
    try:
        power_path = os.path.join(TFR_DIR, POWER_FNAME.format(sub=sub))
        meta_path = os.path.join(TFR_DIR, META_FNAME.format(sub=sub))

        if not os.path.exists(power_path):
            raise FileNotFoundError(f"Missing power file: {power_path}")
        if not os.path.exists(meta_path):
            raise FileNotFoundError(f"Missing meta file: {meta_path}")

        B_global, meta_in, info = compute_B_global_from_saved_tfr(
            power_path=power_path,
            meta_path=meta_path,
            baseline=baseline,
            tol=tol
        )

        out_npy, out_meta = save_B_global(sub, B_global, meta_in, info, OUT_DIR)

        summary[sub] = {
            "out_npy": out_npy,
            "out_meta": out_meta,
            "n_baseline_pts": info["n_baseline_pts"],
            "baseline_start_time": info["times_at_baseline_start"],
            "baseline_end_time": info["times_at_baseline_end"],
            "bglobal_shape": list(B_global.shape),
        }

        print(f"[{sub}] OK  B_global shape={B_global.shape}  baseline_pts={info['n_baseline_pts']}  "
              f"({info['times_at_baseline_start']:.6f}..{info['times_at_baseline_end']:.6f})")

    except Exception as e:
        errors[sub] = str(e)
        print(f"[{sub}] ERROR: {e}")

# Save logs
summary_path = os.path.join(OUT_DIR, "Bglobal_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

if errors:
    errors_path = os.path.join(OUT_DIR, "Bglobal_errors.json")
    with open(errors_path, "w", encoding="utf-8") as f:
        json.dump(errors, f, indent=2, ensure_ascii=False)

print("\nSaved summary:", summary_path)
if errors:
    print("Saved errors :", errors_path)

# 6) QUICK-PLOT: random subject among successful ones
ok_subjects = [s for s in subjects if s in summary]
if len(ok_subjects) > 0:
    sub_qc = random.choice(ok_subjects)
    ppath = os.path.join(TFR_DIR, POWER_FNAME.format(sub=sub_qc))
    mpath = os.path.join(TFR_DIR, META_FNAME.format(sub=sub_qc))

    out_png1, out_png2, qc_info = quick_plot_baseline_distribution(
        sub=sub_qc,
        power_path=ppath,
        meta_path=mpath,
        baseline=baseline,
        tol=tol,
        out_dir=OUT_DIR
    )
    qc_path = os.path.join(OUT_DIR, "quickplot_qc_info.json")
    with open(qc_path, "w", encoding="utf-8") as f:
        json.dump({"subject": sub_qc, **qc_info, "plot1": out_png1, "plot2": out_png2},
                  f, indent=2, ensure_ascii=False)

    print("Saved QC plots:")
    print(" ", out_png1)
    print(" ", out_png2)
    print("Saved QC info :", qc_path)
else:
    print("\nNo successful subjects to QC-plot.")

In [ ]:
#compute Target TFR
import os
import json
import numpy as np
import mne

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

TARG_EPOCH_DIR = r"F:\target_epoch"
TARG_FNAME = "{sub}_targ-epo.fif"

# Cue-derived baseline files
B_DIR = r"F:\cue_global_baseline_Bglobal"
B_FNAME = "{sub}_Bglobal_chxf.npy"
B_META_FNAME = "{sub}_Bglobal_meta.npz"

CUE_TFR_DIR = r"F:\cue_single_trial_TFR_noBaeslineCorrection"
CUE_META_FNAME = "{sub}_cue_meta_noBC.npz"

OUT_DIR = r"F:\target_TFR"
os.makedirs(OUT_DIR, exist_ok=True)

use_fft = True
decim = 1

tmin_targ = -0.2
tmax_targ = 1.0

verbose_mne = False

def load_subject_tfr_params_from_cue_meta(sub: str):
    """Load freqs + n_cycles (and optionally sfreq) from your saved cue meta."""
    meta_path = os.path.join(CUE_TFR_DIR, CUE_META_FNAME.format(sub=sub))
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Missing cue meta for TFR params: {meta_path}")

    meta = np.load(meta_path, allow_pickle=True)
    freqs = meta["freqs"].astype(float)

    # n_cycles saved as either scalar or array
    n_cycles = meta["n_cycles"]
    if np.ndim(n_cycles) == 0:
        n_cycles = float(n_cycles)
    else:
        n_cycles = n_cycles.astype(float)

    return freqs, n_cycles


def save_target_tfr_db(sub: str, tfr_db: mne.time_frequency.EpochsTFR, b_meta, out_dir: str):
    power_path = os.path.join(out_dir, f"{sub}_targ_power_dB.npy")
    meta_path = os.path.join(out_dir, f"{sub}_targ_meta_dB.npz")

    np.save(power_path, tfr_db.data.astype(np.float32))

    meta = dict(
        subject=sub,
        sfreq=float(tfr_db.info["sfreq"]),
        ch_names=np.array(tfr_db.ch_names, dtype=object),
        freqs=tfr_db.freqs.astype(np.float64),
        times=tfr_db.times.astype(np.float64),
        method="morlet",
        average=False,
        return_itc=False,
        use_fft=bool(use_fft),
        decim=int(decim),
        tmin=float(tfr_db.times[0]),
        tmax=float(tfr_db.times[-1]),
        data_shape=np.array(tfr_db.data.shape, dtype=np.int64),
        baseline_reference="cue_global_B_global(ch,f)",
        bglobal_subject=sub,
        bglobal_baseline=b_meta["baseline"] if "baseline" in b_meta else np.array([-0.2, 0.0]),
        bglobal_n_baseline_pts=b_meta["n_baseline_pts"] if "n_baseline_pts" in b_meta else -1,
        note="Target single-trial TFR computed on [0,1]s, then dB-corrected by cue-derived B_global(ch,f): 10*log10(P/B)",
    )
    np.savez(meta_path, **meta)
    return power_path, meta_path

#Main loop
errors = {}
saved = []

print("=== Target single-trial TFR + cue-global baseline dB correction ===")
print("Target epochs dir:", TARG_EPOCH_DIR)
print("B_global dir     :", B_DIR)
print("Output dir       :", OUT_DIR)
print(f"Target window    : {tmin_targ} .. {tmax_targ} s")
print(f"use_fft={use_fft}, decim={decim}")
print("---------------------------------------------------------------")

for sub in subjects:
    try:
        # Paths
        targ_path = os.path.join(TARG_EPOCH_DIR, TARG_FNAME.format(sub=sub))
        b_path = os.path.join(B_DIR, B_FNAME.format(sub=sub))
        b_meta_path = os.path.join(B_DIR, B_META_FNAME.format(sub=sub))

        if not os.path.exists(targ_path):
            raise FileNotFoundError(f"Missing target epochs: {targ_path}")
        if not os.path.exists(b_path):
            raise FileNotFoundError(f"Missing B_global npy: {b_path}")
        if not os.path.exists(b_meta_path):
            raise FileNotFoundError(f"Missing B_global meta: {b_meta_path}")

        # Load B_global and its metadata (channel list lives here)
        B_global = np.load(b_path).astype(np.float64)  # (n_ch, n_freq)
        b_meta = np.load(b_meta_path, allow_pickle=True)
        b_ch_names = list(b_meta["ch_names"])
        b_freqs = b_meta["freqs"].astype(float)

        # Load TFR params from cue meta to keep consistency
        freqs, n_cycles = load_subject_tfr_params_from_cue_meta(sub)

        # Consistency check: freqs should match B_global freqs
        if freqs.shape != b_freqs.shape or not np.allclose(freqs, b_freqs):
            raise ValueError(
                f"[{sub}] freqs mismatch between cue_meta_noBC and B_global_meta.\n"
                f"cue freqs: {freqs[:5]}..{freqs[-5:]}\n"
                f"B    freqs: {b_freqs[:5]}..{b_freqs[-5:]}"
            )

        # Load target epochs
        print(f"\n[{sub}] Loading target epochs ...")
        epochs_targ = mne.read_epochs(targ_path, preload=True, verbose=verbose_mne)

        # Pick + reorder channels to match B_global exactly
        missing = [ch for ch in b_ch_names if ch not in epochs_targ.ch_names]
        if missing:
            raise RuntimeError(f"[{sub}] Target epochs missing channels required by B_global: {missing}")

        # This reorders and drops extra channels (e.g., EOG/ECG) safely
        epochs_targ = epochs_targ.copy().pick_channels(b_ch_names, ordered=True)

        if len(epochs_targ) == 0:
            raise RuntimeError(f"[{sub}] Target epochs has 0 trials.")

        # Ensure target window exists in epochs time axis
        if float(epochs_targ.tmax) < tmax_targ - 1e-12:
            raise ValueError(f"[{sub}] epochs_targ.tmax={epochs_targ.tmax} < required tmax={tmax_targ}")
        if float(epochs_targ.tmin) > tmin_targ + 1e-12:
            # If your epochs start >0, that's fine as long as Lthey cover 0..1; but warn.
            print(f"[{sub}] WARNING: epochs_targ.tmin={epochs_targ.tmin:.3f} > {tmin_targ:.3f}. Check alignment.")

        print(f"[{sub}] epochs_targ: n_trials={len(epochs_targ)}, tmin={epochs_targ.tmin:.3f}, tmax={epochs_targ.tmax:.3f}, n_ch={len(epochs_targ.ch_names)}")

        # Compute TFR ONLY on 0..1s (so your unusable -0.2..0 never enters)
        print(f"[{sub}] Computing target single-trial TFR on {tmin_targ}..{tmax_targ} s ...")
        power_targ = epochs_targ.compute_tfr(
            method="morlet",
            freqs=freqs,
            n_cycles=n_cycles,  #Change所在
            tmin=tmin_targ,
            tmax=tmax_targ,
            average=False,
            return_itc=False,
            decim=1,
            use_fft=use_fft,
            verbose=verbose_mne,
        )
        # Apply dB correction using B_global(ch,f)
        # Broadcast B_global -> (1, n_ch, n_freq, 1)
        eps = np.finfo(float).eps
        B = np.maximum(B_global, eps)

        power_targ_db = power_targ.copy()
        power_targ_db.data = 10.0 * np.log10(power_targ_db.data / B[None, :, :, None])

        # Save
        out_power, out_meta = save_target_tfr_db(sub, power_targ_db, b_meta, OUT_DIR)
        saved.append(sub)

        print(f"[{sub}] Saved:\n  {out_power}\n  {out_meta}")
        print(f"[{sub}] TFR(dB) shape: {power_targ_db.data.shape} (n_trials, n_ch, n_freq, n_times)")
        print(f"[{sub}] times range: {float(power_targ_db.times[0])} .. {float(power_targ_db.times[-1])}")

    except Exception as e:
        errors[sub] = str(e)
        print(f"[{sub}] ERROR: {e}")

In [ ]:
#check epoch -> TFR event_id
power_targ.events
power_targ.event_id
print(np.load(meta_path).files)

In [ ]:
#新基线校准之后的Target single trial TFR
#F:\target_TFR\sub_m_1_02_targ_power_dB.npy
#F:\target_TFR\sub_m_1_02_targ_meta_dB.npz
#target epoch file name model:sub_m_1_02_targ-epo.fif
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import os
os.environ["QT_API"] = "pyqt5"
%matplotlib qt

subjects = ["sub_m_1_02", "sub_m_1_06", "sub_m_1_10","sub_m_1_14","sub_m_1_18","sub_m_1_22","sub_m_1_26","sub_m_1_30","sub_m_1_34","sub_m_1_38","sub_m_1_42","sub_m_1_46",
            "sub_m_2_04","sub_m_2_08","sub_m_2_12","sub_m_2_16","sub_m_2_20","sub_m_2_24","sub_m_2_28","sub_m_2_32","sub_m_2_36","sub_m_2_40","sub_m_2_44",
            "sub_p_1_01","sub_p_1_05","sub_p_1_09","sub_p_1_13","sub_p_1_17","sub_p_1_21","sub_p_1_25","sub_p_1_29","sub_p_1_33","sub_p_1_35","sub_p_1_37","sub_p_1_41","sub_p_1_45",
            "sub_p_2_03","sub_p_2_07","sub_p_2_11","sub_p_2_15","sub_p_2_19","sub_p_2_27","sub_p_2_31","sub_p_2_39","sub_p_2_43","sub_p_2_47"]

OUT_DIR = r"F:/target_TFR"
EPO_DIR = r"F:/target_epoch" 

conds = {"int_rep": "targ/int/rep/sat", "int_swi": "targ/int/swi/sat", "ext_rep": "targ/ext/rep/sat","ext_swi": "targ/ext/swi/sat"}

bands = {"Low Beta": (13, 16),"Beta": (16, 28),"alpha": (8, 15)}

BIN = 0.25
N_PERM = 1000
ALPHA = 0.05
TAIL = 0
SEED = 42

def safe_cond_indices(epochs, cond):
    cond_sel = epochs[cond].selection
    idx = np.flatnonzero(np.isin(epochs.selection, cond_sel))
    return idx

def cond_mean(X, epochs, cond):
    idx = safe_cond_indices(epochs, cond)
    if len(idx) == 0:
        raise RuntimeError(f"No trials for {cond}")
    return X[idx].mean(axis=0)  # (ch, freq, time)

def band_average(M, freqs, f_lo, f_hi):
    fmask = (freqs >= f_lo) & (freqs <= f_hi)
    if not np.any(fmask):
        raise RuntimeError(f"No freq bins in [{f_lo}, {f_hi}]")
    return M[:, fmask, :].mean(axis=1)  # (ch, time)

def build_subject_effects(subj, freqs_ref=None, times_ref=None, ch_ref=None):
    power_path = os.path.join(OUT_DIR, f"{subj}_targ_power_dB.npy")
    meta_path  = os.path.join(OUT_DIR, f"{subj}_targ_meta_dB.npz")
    epo_path   = os.path.join(EPO_DIR, f"{subj}_targ-epo.fif") 

    if not (os.path.exists(power_path) and os.path.exists(meta_path) and os.path.exists(epo_path)):
        return None, None, None, "missing file(s)"

    X = np.load(power_path, mmap_mode="r")  # (trial, ch, freq, time)
    meta = np.load(meta_path, allow_pickle=True)
    ch_names = meta["ch_names"].tolist()
    freqs = meta["freqs"]
    times = meta["times"]

    # axis consistency check
    if freqs_ref is not None:
        if len(freqs) != len(freqs_ref) or not np.allclose(freqs, freqs_ref):
            return None, None, None, "freq mismatch"
    if times_ref is not None:
        if len(times) != len(times_ref) or not np.allclose(times, times_ref):
            return None, None, None, "time mismatch"
    if ch_ref is not None:
        if ch_names != ch_ref:
            return None, None, None, "channel order mismatch"

    epochs = mne.read_epochs(epo_path, preload=True)
    epochs = epochs.crop(tmin=float(times[0]), tmax=float(times[-1]))

    if X.shape[0] != len(epochs):
        return None, None, None, f"trial count mismatch: X={X.shape[0]} vs epochs={len(epochs)}"

    # condition means: (ch,freq,time)
    M = {k: cond_mean(X, epochs, c) for k, c in conds.items()}

    # effects in full TF (ch,freq,time)
    INT = 0.5 * (M["int_rep"] + M["int_swi"])
    EXT = 0.5 * (M["ext_rep"] + M["ext_swi"])
    D_int_ext = INT - EXT

    SWI = 0.5 * (M["int_swi"] + M["ext_swi"])
    REP = 0.5 * (M["int_rep"] + M["ext_rep"])
    D_swi_rep = SWI - REP

    D_inter = (M["int_swi"] - M["int_rep"]) - (M["ext_swi"] - M["ext_rep"])

    out = {
        "rule_switch_int_ext": D_int_ext,
        "att_switch_swi_rep": D_swi_rep,
        "interaction": D_inter,
    }
    return out, freqs, times, None

def make_time_bins(times, bin_s=0.25):
    t0, t1 = float(times[0]), float(times[-1])
    edges = np.arange(t0, t1 + 1e-9, bin_s)
    if edges[-1] < t1:
        edges = np.append(edges, t1)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mask = (times >= a) & (times < b) if b < t1 else (times >= a) & (times <= b)
        if mask.any():
            bins.append((a, b, mask))
    return bins

def run_cluster_time_ch(X_subj_ch_time, info, adjacency, alpha=0.05, n_perm=2000, tail=0, seed=42):
    # MNE expects (n_samples, n_times, n_space) for spatio-temporal with adjacency over space
    X_st = np.transpose(X_subj_ch_time, (0, 2, 1))  # -> (subj, time, ch)

    T_obs, clusters, pvals, H0 = mne.stats.spatio_temporal_cluster_1samp_test(
        X_st,
        adjacency=adjacency,
        n_permutations=n_perm,
        threshold=None,   # uses t-threshold internally (like your logs showed ~2.01)
        tail=tail,
        out_type="mask",
        seed=seed,
        verbose=True,
    )

    # Build sig mask over (time,ch)
    sig = np.zeros(T_obs.shape, dtype=bool)  # (time,ch)
    for cl, p in zip(clusters, pvals):
        if p < alpha:
            sig |= cl  # cl is boolean mask if out_type="mask"

    # convert to (ch,time) for topomap usage
    sig_ch_time = sig.T  # (ch,time)
    return T_obs, clusters, pvals, sig_ch_time

#Load one subject to anchor info/adjacency + axes
anchor = subjects[0]
epo_anchor = os.path.join(EPO_DIR, f"{anchor}_targ-epo.fif")
epochs_anchor = mne.read_epochs(epo_anchor, preload=True)
info = epochs_anchor.copy().pick_types(eeg=True).info

adjacency, ch_names_adj = mne.channels.find_ch_adjacency(info, ch_type="eeg")
print("[adjacency] shape:", adjacency.shape)

# 2) Build subject stacks for each effect+band
effects = ["att_switch_swi_rep", "rule_switch_int_ext", "interaction"]

# store: band -> effect -> list of (ch,time)
stack = {band: {eff: [] for eff in effects} for band in bands.keys()}

freqs_ref = None
times_ref = None
ch_ref = None
kept = 0

for subj in subjects:
    out, freqs, times, err = build_subject_effects(subj, freqs_ref, times_ref, ch_ref)
    if err is not None:
        print("SKIP", subj, "->", err)
        continue

    # lock references on first kept subject
    if freqs_ref is None:
        freqs_ref = freqs.copy()
        times_ref = times.copy()
        meta0 = np.load(os.path.join(OUT_DIR, f"{subj}_targ_meta_dB.npz"), allow_pickle=True)
        ch_ref = meta0["ch_names"].tolist()

    # band-average and collect
    for band_name, (f_lo, f_hi) in bands.items():
        for eff in effects:
            D_ch_f_t = out[eff]  # (ch,freq,time)
            D_ch_t = band_average(D_ch_f_t, freqs_ref, f_lo, f_hi)  # (ch,time)
            stack[band_name][eff].append(D_ch_t.astype(np.float32))

    kept += 1

print("Kept subjects:", kept)

# 3) Cluster + plot: many topomaps per time-bin with black dots
time_bins = make_time_bins(times_ref, BIN)
print("n_time_bins:", len(time_bins), "bin_s:", BIN)

for band_name, (f_lo, f_hi) in bands.items():
    all_maps = []
    for eff in effects:
        X_eff = np.stack(stack[band_name][eff], axis=0)  
        all_maps.append(X_eff.mean(axis=0))           
    all_maps = np.concatenate(all_maps, axis=1)
    vmin, vmax = np.percentile(all_maps, [5, 95])

    n_rows = len(effects)
    n_cols = len(time_bins)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"{band_name.upper()} ({f_lo:.0f}-{f_hi:.0f} Hz) — Cluster permutation topomaps",
        y=0.98
    )

    # +1 column reserved for colorbar
    gs = gridspec.GridSpec(
        n_rows,
        n_cols + 1,
        width_ratios=[1] * n_cols + [0.05],
        wspace=0.25,
        hspace=0.3
    )

    #plot
    for r, eff in enumerate(effects):
        X_eff = np.stack(stack[band_name][eff], axis=0)  # (subj,ch,time)

        T_obs, clusters, pvals, sig_ch_time = run_cluster_time_ch(
            X_eff, info, adjacency,
            alpha=ALPHA, n_perm=N_PERM, tail=TAIL, seed=SEED
        )

        mean_ch_t = X_eff.mean(axis=0)  # (ch,time)

        for c, (ta, tb, tmask) in enumerate(time_bins):
            ax = fig.add_subplot(gs[r, c])

            dat = mean_ch_t[:, tmask].mean(axis=1)
            sig_ch = sig_ch_time[:, tmask].any(axis=1)

            mne.viz.plot_topomap(
                dat,
                info,
                axes=ax,
                show=False,
                vlim=(vmin, vmax),
                contours=0,
                sensors=False,
                mask=sig_ch,
                mask_params=dict(
                    markersize=6,
                    markerfacecolor="k",
                    markeredgecolor="k"
                )
            )

            ax.set_title(f"{ta:+.2f}–{tb:+.2f}s", fontsize=8)
            if c == 0:
                ax.set_ylabel(eff, fontsize=10)

    #  colorbar (single, dedicated axis) 
    cax = fig.add_subplot(gs[:, -1])
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Power (baseline-corrected logratio)", rotation=90)

    plt.show()